# A GMSH Manual

We describe first the use of [GMSH](gmsh.info) for the generation of the geometry and the generation of the geometry. We subsequently discuss how to interate over elements in the mesh (subdomain or boundary) that GMSH generates. 

For the **geometry generation**, GMSH can be employed using one of three methods that are described next: 
1. The first method is to build the geometry in GMSH from the ground up using the GMSH primitives for points, lines, surfaces and volumes. This allows to generate 1D, 2D or 3D geometries. This way of working provides detailed control over the geometry definition. It allows to automate the geometry creation through scripting. Two examples are given later in this notebook.  A possible disadvantage is the large overhead for complex geometries. 
2. The second method is create the geometry in dedicated CAD modeling tools, to save the geometry to file and to import the geometry into the GMSH graphical user interface. Examples of public domain CAD modeling tools include [Blender](https://www.blender.org), [FreeCAD](https://www.freecad.org) and [Salome](https://www.salome-platform.org). 
3. The third method is to use GMSH as a plugin for CAD tools as e.g. a [plugin for Blender](https://github.com/blender-for-science/blendmsh), [plugin for FreeCAD](https://wiki.freecad.org/Macro_GMSH) or [plugin for SALOME](https://docs.salome-platform.org/latest/gui/GMSHPLUGIN/index.html). 

For the (un)structured **mesh generation** in 1D, 2D and 3D, GMSH provides various algorithms that are described in the GMSH manual at [GMSH Mesh Module](https://gmsh.info/doc/texinfo/gmsh.html#Choosing-the-right-unstructured-algorithm). In 2D, a mesh consisting of triangles is called a [triangulation](https://en.wikipedia.org/wiki/Triangulation_(geometry)). Often a variant of the [Delaunay algorithm](https://en.wikipedia.org/wiki/Delaunay_triangulation) is used to construct these triangular meshes. We refer to [Mesh Generation](https://en.wikipedia.org/wiki/Mesh_generation) for more information on mesh generation.

<b>Similar Information Provided Elsewhere</b> Information similar to the one provided here can be found at
1. [GMSH](gmsh.info)
2. [FerriteGmsh.jl](https://github.com/Ferrite-FEM/FerriteGmsh.jl)
3. [GridapGmsh.jl](https://github.com/gridap/GridapGmsh.jl)
4. [Dolfin Project](https://docs.fenicsproject.org/dolfinx/v0.11.0.post0/python/demos/demo_gmsh.html) 

## Import Packages 

In [1]:
try
    using Gmsh: Gmsh, gmsh
catch
    using gmsh
end 

using GR 
using LinearAlgebra
using SparseArrays

using Test 

using Plots

## Section 1: Introduction 

Many of the tutorials that follow consist of the following eight parts: 

1. initialize GMSH;
2. set global options; 
3. construct the geometry; 
4. synchronize the CAD model;
5. define physical groups for (part of the) domain and (part of the) boundary;
6. generate the mesh;
7. view mesh in graphical user interface (GUI) and write mesh to file; 
8. finalize GMSH; 

More later.  

## Section2: Low-Level Geometry Definition using GMSH Primitives 

Here we generate the mesh using low-level GMSH primitives. 

### Section 1.2: Unit Square Tutorial 

Geometry definition and mesh generation of square domain where $0 \leq x \leq 1$ and $0 \leq y \leq 1$. 

The code that follows performs the followings steps. First the <b>geometry</b> on the unit square geometry is defined, then the geometry model is synchronized, then physical groups are added and finally the <b>mesh</b> is generated. 

<b>Geometry Definition</b> 

First the <b>geometry</b> is generated in the following five steps:
1. four corner points of the square are defined. The points are defined by their $(x,y,z)$-coordinates. For two-dimensional geometries, the $z$-coordinate set to zero. The points are labeled as 1 through 4. See docs of function <i>gmsh.model.geo.addPoint()</i> for more information;
2. four lines are defined as the edges of the square are defined by connecting previously defined points. Edges are formed by connecting points pairwise. The lines are given a start and end point. The lines are thus oriented. GMSH distinguishes between tags given to points (entities of dimensions zero), tags given to lines (entities of dimension one), tags given to surfaces (entities of dimension two) and tags given to volumes (entities of dimension three). GMSH uses the convention (dim,tag) to distinguish various entities. The edges are labeled as 1 through 4. See docs of function <i>gmsh.model.geo.addLine()</i> for more information;  
3. the boundary of the square is defined by a loop connecting the four edges. The orientation of the edges given an orientation to the loop. The loop is oriented such that an imaginary observer walking on the loop finds the domain on his left-hand side. The loop is labeled as 1. See docs of function <i>gmsh.model.geo.addCurveLoop()</i> for more information;   
4. the surface of the square is defined by the loop. It is on this square that the mesh generation will take place. This square is labeled as 1. See docs of function <i>gmsh.model.geo.addPlaneSurface()</i> for more information;

<b>Remark on the labeling of points, edges and surfaces</b> 

GMSH distinguishes points, lines, surfaces and volumes as entities of dimension 0, 1, 2 and 3, respectively. GMSH recognizes entities using (dimensions,label)-pairs. Entities of the same dimension therefore should have unique labels. Nothing, however, prevents entities with different dimension to share the same label;

After the geometry definition, the geometry model is synchronized using the function <i>gmsh.model.geo.synchronize()</i> and physical groups for the boundary and the interior of the domain are added to the model using the function <i>gmsh.model.addPhysicalGroup</i>. 

<b>Remark on adding physical groups</b>

We assign physical groups on the subdomain to allow to distinguish e.g. the domain of the coil, the ferromagnetic material and the coils. We assign physical groups to the boundaries to allow the definition of the boundary conditions. In the unit square tutorial, we assign distinct labels to the bottom, left, top and bottom side of the boundary. The graphical user interface of GMSH can be used to find the labeling of edges and subdomains. Open the so-called Quick Access Menu and choose <i>All Geometry Options...</i>. This allows up to assign distinct boundary conditions to these sides. 

<b>Mesh Generation</b>

After the geometry is defined, the <b>mesh</b> on the geometry is generated using the function <i>gmsh.model.mesh.generate(2)</i>, where the input refers to two-dimensional mesh generation. By default, the mesh is generated by first meshing the four edges of the square. The mesh is subsequently propagated towards the interior of the square. The mesh density is controlled by the parameter lc. 

The mesh is optionally written to an output file and visualized using the GUI.

<b>Remark on the number of elements provided as output after mesh generation</b>

In providing a number of elements, GMSH aggregates the 1D elements on the boundary and the 2D elements on the interior of the domain. The number of 2D elements on the interior of the domain is thus given by the output of <i>gmsh.model.mesh.generate(2)</i> minus the output of <i>gmsh.model.mesh.generate(1)</i>. 

<b>Exercises</b>
1. find the online documentation to the <i>gmsh.option</i> functions; 
2. consider a mesh with a sufficiently small number of elements. Draw a graph of the graph of the node (point) - element (triangle) connectivity. Possibly use [GraphMakie.jl](https://graph.makie.org/stable/);
3. as above, now for the node - edge connectivity;  

In [2]:
?gmsh.model.geo.addLine

```
gmsh.model.geo.addLine(startTag, endTag, tag = -1)
```

Add a straight line segment in the built-in CAD representation, between the two points with tags `startTag` and `endTag`. If `tag` is positive, set the tag explicitly; otherwise a new tag is selected automatically. Return the tag of the line.

Return an integer.

Types:

  * `startTag`: integer
  * `endTag`: integer
  * `tag`: integer


In [3]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal",3) # make more verbose 
gmsh.option.setNumber("Mesh.Algorithm",6)   # choose mesh algorithm 
gmsh.model.add("square")                    # give name to model 

#..3/8: generate geometry 
#..set mesh density parameter 
lc = .1
#..define four points via (x,y,z) coordinates 
p1 = gmsh.model.geo.addPoint(0, 0, 0, lc, 1)
p2 = gmsh.model.geo.addPoint(1., 0,  0, lc, 2)
p3 = gmsh.model.geo.addPoint(1., 1., 0, lc, 3)
p4 = gmsh.model.geo.addPoint(0, 1., 0, lc, 4)
#..define four edges by connecting point labels pairwise  
l1 = gmsh.model.geo.addLine(1, 2, 1)
l2 = gmsh.model.geo.addLine(2, 3, 2)
l3 = gmsh.model.geo.addLine(3, 4, 3)
l4 = gmsh.model.geo.addLine(4, 1, 4)
#..define curved loop by connecting four edge labels  
loop = gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1)
#..define surface by curved loop 
surf = gmsh.model.geo.addPlaneSurface([1], 1)

#..4/8: synchronize the CAD model 
gmsh.model.geo.synchronize()

#..5/8: assign physical groups for the four boundaries and the interior 
gmsh.model.addPhysicalGroup(1, [l1], -1, "bottom")
gmsh.model.addPhysicalGroup(1, [l2], -1, "right")
gmsh.model.addPhysicalGroup(1, [l3], -1, "top")
gmsh.model.addPhysicalGroup(1, [l4], -1, "left")
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

#..6/8: generate two-dimensional mesh 
gmsh.model.mesh.generate(2)

#..7/8: write mesh to mesh and visualize the mesh  
#..if true, write mesh to file for further processing
gmsh.option.setNumber("Mesh.Format", 16)
if (true) gmsh.write("square.msh") end 
#..if true, visualize mesh through the GUI 
if (true) gmsh.fltk.run() end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000511583s, CPU 0.000102s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00330779s, CPU 0.001344s)
Info    : 142 nodes 286 elements
Info    : Writing 'square.msh'...
Info    : Done writing 'square.msh'
-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : MacOSX-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugin

2026-06-17 17:42:22.176 julia[75648:6884697] +[IMKClient subclass]: chose IMKClient_Modern
2026-06-17 17:42:22.176 julia[75648:6884697] +[IMKInputSession subclass]: chose IMKInputSession_Modern


### Section 2.2: Exercise: General Quadrilateral 

Change the coordinate of one of the four points of the square and regenerate the mesh (thus practising with points, edges, closed loops and surfaces);

In [4]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal",1) # make more verbose 
gmsh.option.setNumber("Mesh.Algorithm",6) # choose mesh algorithm 
gmsh.model.add("square")

#..3/8: generate geometry 
#..set mesh density parameter 
lc = .1
#..define four points via (x,y,z) coordinates 
p1 = gmsh.model.geo.addPoint(0, 0, 0, lc, 1)
p2 = gmsh.model.geo.addPoint(1., 0,  0, lc, 2)
p3 = gmsh.model.geo.addPoint(1.5, 0.5, 0, lc, 3) #..p3 CHANGED to (x,y) (1.5,0.5) 
p4 = gmsh.model.geo.addPoint(0, 1., 0, lc, 4)
#..define four edges by connecting point labels pairwise  
l1 = gmsh.model.geo.addLine(1, 2, 1)
l2 = gmsh.model.geo.addLine(2, 3, 2)
l3 = gmsh.model.geo.addLine(3, 4, 3)
l4 = gmsh.model.geo.addLine(4, 1, 4)
#..define curved loop by connecting four edge labels  
loop = gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1)
#..define surface by curved loop 
surf = gmsh.model.geo.addPlaneSurface([1], 1)

#..4/8: synchronize the CAD model 
gmsh.model.geo.synchronize()

#..5/8: assign physical groups
gmsh.model.addPhysicalGroup(1, [l1], -1, "bottom")
gmsh.model.addPhysicalGroup(1, [l2], -1, "right")
gmsh.model.addPhysicalGroup(1, [l3], -1, "top")
gmsh.model.addPhysicalGroup(1, [l4], -1, "left")
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

#..6/8: generate two-dimensional mesh 
gmsh.model.mesh.generate(2)

#..7/8: write mesh to mesh and visualize the mesh  
#..if true, write mesh to file for further processing
gmsh.option.setNumber("Mesh.Format", 16)
if (true) gmsh.write("square.msh") end 
#..if true, visualize mesh through the GUI 
if (true) gmsh.fltk.run() end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000488458s, CPU 0.000475s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00517438s, CPU 0.005133s)
Info    : 153 nodes 308 elements
Info    : Writing 'square.msh'...
Info    : Done writing 'square.msh'
-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : MacOSX-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugin

2026-06-11 09:01:11.977 julia[62588:5645738] +[IMKClient subclass]: chose IMKClient_Modern
2026-06-11 09:01:11.977 julia[62588:5645738] +[IMKInputSession subclass]: chose IMKInputSession_Modern


false

### Section 2.3: Exercise: Regular Pentagon of Radius 1 

Extend the code to the generation of a mesh on a pentagon (thus practising with points, edges, closed loops and surfaces). 

In the solution detailed below a regular polygon with $n$ vertices is generated. We show how the generation of points and edges can be placed within a for-loop.  

#### Build a geometry model for regular polygon and use it for pentagon 

A regular pentagon has 5 vertices equally spaced on a circle of radius $r = 1$. Any point on a circle can be described by the parametric equations:

$$x_i = r \cos(\theta_i), \quad y_i = r \sin(\theta_i)$$

where $\theta_i$ is the angle of the $i$-th vertex measured from the positive $x$-axis.

#### Choosing the Angles

For $n$ equally spaced points, the angular spacing between consecutive vertices is:

$$\Delta\theta = \frac{2\pi}{n}$$

For $n = 5$, this gives $\Delta\theta = \frac{2\pi}{5} = 72\deg$.

We want the first vertex at the top of the circle (12 o'clock), which corresponds to $\theta = \frac{\pi}{2}$. So the angle of the i-th vertex is:

$$\theta_i = \frac{\pi}{2} + \frac{2\pi(i-1)}{n}, \quad i = 1, 2, \ldots, n$$

This gives:

| $i$ | $\theta_i$ (rad) | $\theta_i$ (deg) | $x_i = \cos\theta_i$ | $y_i = \sin\theta_i$ |
|-----|-------------------|-------------------|-----------------------|-----------------------|
| 1   | $\frac{\pi}{2}$                   | 90deg  | 0.0    | 1.0    |
| 2   | $\frac{\pi}{2} + \frac{2\pi}{5}$  | 162deg | -0.951 | 0.309  |
| 3   | $\frac{\pi}{2} + \frac{4\pi}{5}$  | 234deg | -0.588 | -0.809 |
| 4   | $\frac{\pi}{2} + \frac{6\pi}{5}$  | 306deg | 0.588  | -0.809 |
| 5   | $\frac{\pi}{2} + \frac{8\pi}{5}$  | 378deg | 0.951  | 0.309  |

#### In Julia Code

The loop translates the math directly:
```julia
n = 5
for i in 1:n
    angle = π/2 + 2π*(i-1)/n    # θ_i from the formula above
    x = cos(angle)               # x_i = cos(θ_i)
    y = sin(angle)               # y_i = sin(θ_i)
    gmsh.model.geo.addPoint(x, y, 0, lc, i)
end
```

#### Connecting the Vertices

Each edge of the pentagon connects vertex $i$ to vertex $i+1$, with the last edge wrapping from vertex $n$ back to vertex $1$ that is last to first vertex. Mathematically it is

$\text{edge}_i: \quad P_i$ to $P_{(i \mod n) + 1}$

The modular arithmetic handles the wrap around. For $n = 5$:

| Edge $i$ | From $P_i$ | $i \mod n$ | $(i \mod n) + 1$ | To $P$ |
|----------|------------|------------|-------------------|--------|
| 1        | $P_1$      | 1          | 2                 | $P_2$  |
| 2        | $P_2$      | 2          | 3                 | $P_3$  |
| 3        | $P_3$      | 3          | 4                 | $P_4$  |
| 4        | $P_4$      | 4          | 5                 | $P_5$  |
| 5        | $P_5$      | 0          | 1                 | $P_1$  |

In code:
```julia
for i in 1:n
    next = (i % n) + 1
    gmsh.model.geo.addLine(i, next, i)
end
```

#### Curve Loop and Surface

The curve loop collects all $n$ edges into one closed boundary. The CCW orientation ensures the enclosed region is on the left side of the walking direction.

The plane surface then tells GMSH: the region bounded by this loop is a 2D domain to be meshed.
```julia
gmsh.model.geo.addCurveLoop(collect(1:n), 1)   # closed boundary
gmsh.model.geo.addPlaneSurface([1], 1)          # 2D region inside
```

#### Generalization

Since n is a variable, changing `n = 5` to any other integer produces a regular polygon with that many sides — hexagon ($n=6$), octagon ($n=8$), etc. As $n \to \infty$, the polygon approximates a circle.

In [8]:
#General Approach
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal",1) # make more verbose 
gmsh.option.setNumber("Mesh.Algorithm",6) # choose mesh algorithm 
gmsh.model.add("Pentagon")

#..3/8: generate geometry 
#..set mesh density parameter 
lc = 0.1

#..define five points via (x,y,z) coordinates 
n = 5  # number of verices 
for i in 1:n
    angle = π/2 + 2π*(i-1)/n          # start from top, go CCW
    x = cos(angle)
    y = sin(angle)
    gmsh.model.geo.addPoint(x, y, 0, lc, i)  # tag = i
end

# adding 5 lines
for i in 1:n
    next = (i % n) + 1                # wraps 5 back to 1
    gmsh.model.geo.addLine(i, next, i)   # tag = i
end

#..define curved loop by connecting four edge labels  
gmsh.model.geo.addCurveLoop(collect(1:n), 1)
#..define surface by curved loop 
surf = gmsh.model.geo.addPlaneSurface([1], 1)

#..4/8: synchronize the CAD model 
gmsh.model.geo.synchronize()

#..5/8: assign physical groups
for i in 1:n
    gmsh.model.addPhysicalGroup(1, [i], -1, "edge_$i")
end
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

#..6/8: generate two-dimensional mesh 
gmsh.model.mesh.generate(2)

#..7/8: write mesh to mesh and visualize the mesh  
#..if true, write mesh to file for further processing
gmsh.option.setNumber("Mesh.Format", 16)
if (true) gmsh.write("pentagon.msh") end 
#..if true, visualize mesh through the GUI 
if (true) gmsh.fltk.run() end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 50%] Meshing curve 3 (Line)
Info    : [ 70%] Meshing curve 4 (Line)
Info    : [ 90%] Meshing curve 5 (Line)
Info    : Done meshing 1D (Wall 0.000183875s, CPU 0.000176s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00457746s, CPU 0.004522s)
Info    : 331 nodes 665 elements
Info    : Writing 'pentagon.msh'...
Info    : Done writing 'pentagon.msh'
-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : MacOSX-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCA

false

### Section 2.4: Exercise: L-shape  

Extend the code to the generation of a mesh on an L-shaped domain (thus practising with points, edges, closed loops and surfaces);

In [12]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal",1) # make more verbose 
gmsh.option.setNumber("Mesh.Algorithm",6) # choose mesh algorithm 
gmsh.model.add("L_shape")

#..3/8: generate geometry 
#..set mesh density parameter 
lc = .1
#..define 6 points via (x,y,z) coordinates 
p1 = gmsh.model.geo.addPoint(0, 0, 0, lc, 1)
p2 = gmsh.model.geo.addPoint(0, 2,  0, lc, 2)
p3 = gmsh.model.geo.addPoint(0.5, 2, 0, lc, 3)
p4 = gmsh.model.geo.addPoint(0.5, 0.5, 0, lc, 4)
p5 = gmsh.model.geo.addPoint(2, 0.5, 0, lc, 5)
p6 = gmsh.model.geo.addPoint(2, 0, 0, lc, 6)
#..define 6 edges by connecting point labels pairwise  
l1 = gmsh.model.geo.addLine(1, 2, 1)
l2 = gmsh.model.geo.addLine(2, 3, 2)
l3 = gmsh.model.geo.addLine(3, 4, 3)
l4 = gmsh.model.geo.addLine(4, 5, 4)
l5 = gmsh.model.geo.addLine(5, 6, 5)
l6 = gmsh.model.geo.addLine(6, 1, 6)
#..define curved loop by connecting four edge labels  
loop = gmsh.model.geo.addCurveLoop([1, 2, 3, 4, 5, 6], 1)
#..define surface by curved loop 
surf = gmsh.model.geo.addPlaneSurface([1], 1)

#..4/8: synchronize the CAD model 
gmsh.model.geo.synchronize()

#..5/8: assign physical groups
gmsh.model.addPhysicalGroup(1, [l1], -1, "bottom")
gmsh.model.addPhysicalGroup(1, [l2], -1, "right")
gmsh.model.addPhysicalGroup(1, [l3], -1, "top")
gmsh.model.addPhysicalGroup(1, [l4], -1, "left")
gmsh.model.addPhysicalGroup(1, [l5], -1, "extra1")
gmsh.model.addPhysicalGroup(1, [l4], -1, "extra2")
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

#..6/8: generate two-dimensional mesh 
gmsh.model.mesh.generate(2)

#..7/8: write mesh to mesh and visualize the mesh  
#..if true, write mesh to file for further processing
gmsh.option.setNumber("Mesh.Format", 16)
if (true) gmsh.write("L_shape.msh") end 
#..if true, visualize mesh through the GUI 
if (true) gmsh.fltk.run() end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 40%] Meshing curve 3 (Line)
Info    : [ 60%] Meshing curve 4 (Line)
Info    : [ 70%] Meshing curve 5 (Line)
Info    : [ 90%] Meshing curve 6 (Line)
Info    : Done meshing 1D (Wall 0.000550834s, CPU 0.000505s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00631604s, CPU 0.00596s)
Info    : 257 nodes 518 elements
Info    : Writing 'L_shape.msh'...
Info    : Done writing 'L_shape.msh'
-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : MacOSX-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONEL

false

### Section 2.5: Exercise: Outer Square with Inner Square Hole   

Extend the code to the generation of a mesh on a square with an inner square removed (thus practising the orientation of the loops). 

In the solution detailed below, the inner and outer boundary are defined in clockwise (CW) and counter-clockwise direction, respectively. 

In [ ]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal", 1)
gmsh.option.setNumber("Mesh.Algorithm", 6)
gmsh.model.add("square_with_hole")

#..3/8: generate geometry 
lc = 0.1

# Outer corners at (0,0), (1,0), (1,1), (0,1)
gmsh.model.geo.addPoint(0, 0, 0, lc, 1)
gmsh.model.geo.addPoint(1, 0, 0, lc, 2)
gmsh.model.geo.addPoint(1, 1, 0, lc, 3)
gmsh.model.geo.addPoint(0, 1, 0, lc, 4)

# Innercorners at (0.3,0.3), (0.7,0.3), (0.7,0.7), (0.3,0.7)
gmsh.model.geo.addPoint(0.3, 0.3, 0, lc, 5)
gmsh.model.geo.addPoint(0.7, 0.3, 0, lc, 6)
gmsh.model.geo.addPoint(0.7, 0.7, 0, lc, 7)
gmsh.model.geo.addPoint(0.3, 0.7, 0, lc, 8)

# Outer lines in CCW order (1-2-3-4-1)
gmsh.model.geo.addLine(1, 2, 1)   # bottom
gmsh.model.geo.addLine(2, 3, 2)   # right
gmsh.model.geo.addLine(3, 4, 3)   # top
gmsh.model.geo.addLine(4, 1, 4)   # left

#  Inner lines in CW order (5-8-7-6-5) 
gmsh.model.geo.addLine(5, 8, 5)   # left side of inner square
gmsh.model.geo.addLine(8, 7, 6)   # top side of inner square
gmsh.model.geo.addLine(7, 6, 7)   # right side of inner square
gmsh.model.geo.addLine(6, 5, 8)   # bottom side of inner square

# outer loop: CCW (tags 1,2,3,4)
gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1)
# inner loop: CW (tags 5,6,7,8)
gmsh.model.geo.addCurveLoop([5, 6, 7, 8], 2)

#For Surfaace geenration both loops are passed first is boundary, second is hole
surf = gmsh.model.geo.addPlaneSurface([1, 2], 1)

#..4/8: synchronize
gmsh.model.geo.synchronize()

#..5/8: physical groups
gmsh.model.addPhysicalGroup(1, [1], -1, "outer_bottom")
gmsh.model.addPhysicalGroup(1, [2], -1, "outer_right")
gmsh.model.addPhysicalGroup(1, [3], -1, "outer_top")
gmsh.model.addPhysicalGroup(1, [4], -1, "outer_left")
gmsh.model.addPhysicalGroup(1, [5, 6, 7, 8], -1, "inner_boundary")
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

#..6/8: generate mesh
gmsh.model.mesh.generate(2)

#..7/8: write to file
if (true) gmsh.write("square_with_hole.msh") end
if (true) gmsh.fltk.run() end

#..8/8: finalize
should_finalize && Gmsh.finalize()

### Section 2.6: Exercise: Square with Subdomains and Inner Boundaries 

Extend the code to the generation of a mesh on a square with an inner square removed (thus practising the orientation of the loops and with the labeling of boundaries and subdomains).

In the solution detailed below, the inner and outer domain receive distinct mesh densities. Furthermore, subdomains, outer boundaries and inner boundaries receive seperate labels.  

In [18]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal",1)
gmsh.option.setNumber("Mesh.Algorithm",6)
gmsh.model.add("concentric-squares")

#..3/8: generate geometry 
# Mesh densities - different for each region
lc_outer = 0.2      # mesh outer region
lc_inner = 0.05     # mesh for inner region

# Outer square points 
p1 = gmsh.model.geo.addPoint(0, 0, 0, lc_outer, 1)
p2 = gmsh.model.geo.addPoint(2, 0, 0, lc_outer, 2)
p3 = gmsh.model.geo.addPoint(2, 2, 0, lc_outer, 3)
p4 = gmsh.model.geo.addPoint(0, 2, 0, lc_outer, 4)

# Inner square points 
p5 = gmsh.model.geo.addPoint(0.5, 0.5, 0, lc_inner, 5)
p6 = gmsh.model.geo.addPoint(1.5, 0.5, 0, lc_inner, 6)
p7 = gmsh.model.geo.addPoint(1.5, 1.5, 0, lc_inner, 7)
p8 = gmsh.model.geo.addPoint(0.5, 1.5, 0, lc_inner, 8)

# Outer square edges
l1 = gmsh.model.geo.addLine(1, 2, 1)
l2 = gmsh.model.geo.addLine(2, 3, 2)
l3 = gmsh.model.geo.addLine(3, 4, 3)
l4 = gmsh.model.geo.addLine(4, 1, 4)

# Inner square edges
l5 = gmsh.model.geo.addLine(5, 6, 5)
l6 = gmsh.model.geo.addLine(6, 7, 6)
l7 = gmsh.model.geo.addLine(7, 8, 7)
l8 = gmsh.model.geo.addLine(8, 5, 8)

# Create curve loops
outer_loop = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4], 1)
inner_loop = gmsh.model.geo.addCurveLoop([l5, l6, l7, l8], 2)

# Create surfaces
# Inner square surface
inner_surface = gmsh.model.geo.addPlaneSurface([inner_loop], 2)

# Outer ring surface (outer loop minus inner loop)
outer_surface = gmsh.model.geo.addPlaneSurface([outer_loop, inner_loop], 1)

#..4/8: synchronize 
gmsh.model.geo.synchronize()

#..5/8: assign physical groups
# Physical groups for boundaries
gmsh.model.addPhysicalGroup(1, [l1], -1, "outer-bottom")
gmsh.model.addPhysicalGroup(1, [l2], -1, "outer-right")
gmsh.model.addPhysicalGroup(1, [l3], -1, "outer-top")
gmsh.model.addPhysicalGroup(1, [l4], -1, "outer-left")

gmsh.model.addPhysicalGroup(1, [l5], -1, "inner-bottom")
gmsh.model.addPhysicalGroup(1, [l6], -1, "inner-right")
gmsh.model.addPhysicalGroup(1, [l7], -1, "inner-top")
gmsh.model.addPhysicalGroup(1, [l8], -1, "inner-left")

# Physical groups for subdomains (different materials/regions)
gmsh.model.addPhysicalGroup(2, [outer_surface], 1, "OuterRegion")
gmsh.model.addPhysicalGroup(2, [inner_surface], 2, "InnerRegion")

#..6/8: generate mesh 
gmsh.model.mesh.generate(2)

#..7/8: write to file
if (true) gmsh.write("concentric-squares.msh") end
if (true) gmsh.fltk.run() end

#..8/8: finalize
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 60%] Meshing curve 5 (Line)
Info    : [ 70%] Meshing curve 6 (Line)
Info    : [ 80%] Meshing curve 7 (Line)
Info    : [ 90%] Meshing curve 8 (Line)
Info    : Done meshing 1D (Wall 0.00068325s, CPU 0.000669s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 2 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0143831s, CPU 0.014262s)
Info    : 827 nodes 1740 elements
Info    : Writing 'concentric-squares.msh'...
Info    : Done writing 'concentric-squares.msh'
-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : MacOSX-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[cont

false

## Section 3: A Closer Look into Mesh Generation 

### Section 1.3: Mesh Generation algorithms  

Provide overview of mesh generation algorithms that GMSH provides.   

### Section 2.3: Mesh Density 

In the unit square tutorial, change the mesh density by changing the value of the parameter lc and regenerate the mesh. Apply different mesh density on one or more point or edges of the square (thus practising with mesh density settings). 

In the solution detailed below, the lower left and lower right corner of the square are meshed finer then the top left and top right cornrer.

In [20]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal", 1)
gmsh.option.setNumber("Mesh.Algorithm", 6)
gmsh.model.add("variable_density")

#..3/8: generate geometry
# two different mesh densities
lc_fine   = 0.05   
lc_coarse = 0.5    

# bottom two points get fine mesh
gmsh.model.geo.addPoint(0, 0, 0, lc_fine,   1)
gmsh.model.geo.addPoint(1, 0, 0, lc_fine,   2)
# top two points get coarse mesh
gmsh.model.geo.addPoint(1, 1, 0, lc_coarse, 3)
gmsh.model.geo.addPoint(0, 1, 0, lc_coarse, 4)

#..define four edges by connecting point labels pairwise  
l1 = gmsh.model.geo.addLine(1, 2, 1)
l2 = gmsh.model.geo.addLine(2, 3, 2)
l3 = gmsh.model.geo.addLine(3, 4, 3)
l4 = gmsh.model.geo.addLine(4, 1, 4)
#..define curved loop by connecting four edge labels  
loop = gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1)
#..define surface by curved loop 
surf = gmsh.model.geo.addPlaneSurface([1], 1)

#..4/8: synchronize the CAD model 
gmsh.model.geo.synchronize()

#..5/8: assign physical groups
gmsh.model.addPhysicalGroup(1, [l1], -1, "bottom")
gmsh.model.addPhysicalGroup(1, [l2], -1, "right")
gmsh.model.addPhysicalGroup(1, [l3], -1, "top")
gmsh.model.addPhysicalGroup(1, [l4], -1, "left")
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

#..6/8: generate two-dimensional mesh 
gmsh.model.mesh.generate(2)

#..7/8: write mesh to mesh and visualize the mesh  
#..if true, write mesh to file for further processing
gmsh.option.setNumber("Mesh.Format", 16)
if (true) gmsh.write("variable_density.msh") end 
#..if true, visualize mesh through the GUI 
if (true) gmsh.fltk.run() end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00494083s, CPU 0.004649s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00252133s, CPU 0.00248s)
Info    : 110 nodes 222 elements
Info    : Writing 'variable_density.msh'...
Info    : Done writing 'variable_density.msh'
-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : MacOSX-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP Op

false

### Section 3.3: Second Order Elements 

In the unit square tutorial, change from first to second order elements. Verify how the information written to file for first order and second order meshes differs;

In [23]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal",3) # make more verbose 
gmsh.option.setNumber("Mesh.ElementOrder", 2)
gmsh.option.setNumber("Mesh.SecondOrderLinear", 1)
gmsh.option.setNumber("Mesh.Algorithm",6)   # choose mesh algorithm 
gmsh.model.add("square")                    # give name to model 

#..3/8: generate geometry 
#..set mesh density parameter 
lc = .1
#..define four points via (x,y,z) coordinates 
p1 = gmsh.model.geo.addPoint(0, 0, 0, lc, 1)
p2 = gmsh.model.geo.addPoint(1., 0,  0, lc, 2)
p3 = gmsh.model.geo.addPoint(1., 1., 0, lc, 3)
p4 = gmsh.model.geo.addPoint(0, 1., 0, lc, 4)
#..define four edges by connecting point labels pairwise  
l1 = gmsh.model.geo.addLine(1, 2, 1)
l2 = gmsh.model.geo.addLine(2, 3, 2)
l3 = gmsh.model.geo.addLine(3, 4, 3)
l4 = gmsh.model.geo.addLine(4, 1, 4)
#..define curved loop by connecting four edge labels  
loop = gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1)
#..define surface by curved loop 
surf = gmsh.model.geo.addPlaneSurface([1], 1)

#..4/8: synchronize the CAD model 
gmsh.model.geo.synchronize()

#..5/8: assign physical groups for the four boundaries and the interior 
gmsh.model.addPhysicalGroup(1, [l1], -1, "bottom")
gmsh.model.addPhysicalGroup(1, [l2], -1, "right")
gmsh.model.addPhysicalGroup(1, [l3], -1, "top")
gmsh.model.addPhysicalGroup(1, [l4], -1, "left")
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

#..6/8: generate two-dimensional mesh 
gmsh.model.mesh.generate(1)

#..7/8: write mesh to mesh and visualize the mesh  
#..if true, write mesh to file for further processing
gmsh.option.setNumber("Mesh.Format", 16)
if (true) gmsh.write("square.msh") end 
#..if true, visualize mesh through the GUI 
if (false) gmsh.fltk.run() end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000556084s, CPU 0.000504s)
Info    : Meshing order 2 (curvilinear off)...
Info    : [  0%] Meshing curve 1 order 2
Info    : [ 30%] Meshing curve 2 order 2
Info    : [ 50%] Meshing curve 3 order 2
Info    : [ 70%] Meshing curve 4 order 2
Info    : [ 90%] Meshing surface 1 order 2
Info    : Done meshing order 2 (Wall 0.000165084s, CPU 0.000108s)
Info    : 80 nodes 44 elements
Info    : Writing 'square.msh'...
Info    : Done writing 'square.msh'


false

### Section 4.3: Quadrilateral Mesh  

In the unit square tutorial, modify the code to the generation of a mesh consisting of quadrilaterals instead of triangles. Use either the function <i>gmsh.model.mesh.setRecombine(2, surf)</i> or or <i>gmsh.model.mesh.setRecombine(2, surf)</i> ). Verify how the information written to file for triangular and quadrilateral meshes differs;

In [24]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal", 1)
gmsh.option.setNumber("Mesh.Algorithm", 6)
gmsh.model.add("quad_square")

#..3/8: generate geometry (unit square 0 <= x <= 1 and 0 <= y <= 1)
#....set mesh size parameter 
lc = 0.1

gmsh.model.geo.addPoint(0, 0, 0, lc, 1)
gmsh.model.geo.addPoint(1, 0, 0, lc, 2)
gmsh.model.geo.addPoint(1, 1, 0, lc, 3)
gmsh.model.geo.addPoint(0, 1, 0, lc, 4)

gmsh.model.geo.addLine(1, 2, 1)
gmsh.model.geo.addLine(2, 3, 2)
gmsh.model.geo.addLine(3, 4, 3)
gmsh.model.geo.addLine(4, 1, 4)

gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1)
surf = gmsh.model.geo.addPlaneSurface([1], 1)

#..4/8: synchronize
gmsh.model.geo.synchronize()

#..5/8: physical groups
gmsh.model.addPhysicalGroup(1, [1], -1, "bottom")
gmsh.model.addPhysicalGroup(1, [2], -1, "right")
gmsh.model.addPhysicalGroup(1, [3], -1, "top")
gmsh.model.addPhysicalGroup(1, [4], -1, "left")
gmsh.model.addPhysicalGroup(2, [surf], -1, "omega")

# Tell GMSH to recombine triangles into quads on this surface
gmsh.model.mesh.setRecombine(2, surf)

#..6/8: generate mesh
gmsh.model.mesh.generate(2)

#..7/8: write to file
if (true) gmsh.write("quad_square.msh") end
if (true) gmsh.fltk.run() end

#..8/8: finalize
should_finalize && Gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000463708s, CPU 0.000348s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Blossom: 343 internal 40 closed
Info    : Blossom recombination completed (Wall 0.00247346s, CPU 0.002211s): 119 quads, 0 triangles, 0 invalid quads, 0 quads with Q < 0.1, avg Q = 0.817496, min Q = 0.506358
Info    : Done meshing 2D (Wall 0.00591492s, CPU 0.005387s)
Info    : Meshing order 2 (curvilinear off)...
Info    : [  0%] Meshing curve 1 order 2
Info    : [ 30%] Meshing curve 2 order 2
Info    : [ 50%] Meshing curve 3 order 2
Info    : [ 70%] Meshing curve 4 order 2
Info    : [ 90%] Meshing surface 1 order 2
Info    : Done meshing order 2 (Wall 0.00182475s, CPU 0.000382s)
Info    : 517 nodes 163 elements
Info    : Writing 'quad_square.msh'...
Info    :

false

## Section 4: High-Level Geometry Definition using Open-Cascasde

Here we discuss the use of Open-Cascade to define the geometry. We thus provide an alternative for the low level GMSH geometry promimitives. See [GMSH OpenCASCADE CAD kernel functions](https://gmsh.info/doc/texinfo/gmsh.html#Namespace-gmsh_002fmodel_002focc) for more details. 

We use the function <i>add_rectangle(x,y,z,dx,dy,tag=-1,roundedRadius=0.)</i> to define the rectangle with lower left corner at $(x,y,z)$ and upper right corner at $(x+dx,y+dy,z+dz)$. 

<b>Exercises</b>
1. find alternatives to <i>gmsh.option.setNumber("Mesh.MeshSizeMax",0.1)</i> to control the global (and local?) mesh density;
2. find the functionality to determine to which physical group an element on the subdomain (a triangle) or an element on the boundary (an edge) belongs;  

### Section 1.4: Unit Square (Disk) Tutorial using Open-Cascade 

In [14]:
Gmsh.finalize()

In [ ]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.set_number("General.Verbosity",3 )         # make more verbose
gmsh.option.setNumber("Mesh.Algorithm",11)             # choose mesh algorithm
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature",2)
gmsh.option.setNumber("Mesh.MeshSizeMax",0.1)          # set mesh density 

#..3/8: generate geometry (unit square 0 <= x <= 1 and 0 <= y <= 1)
surface_tag = gmsh.model.occ.add_rectangle(0,0,0,1,1)
if (true) surface_tag = gmsh.model.occ.add_disk(0, 0, 0, 1, 1) end 

#..4/8: synchronize
gmsh.model.occ.synchronize()

#..5/8: physical groups
gmsh.model.addPhysicalGroup(1, [1], -1, "bottom")
gmsh.model.addPhysicalGroup(1, [2], -1, "right")
gmsh.model.addPhysicalGroup(1, [3], -1, "top")
gmsh.model.addPhysicalGroup(1, [4], -1, "left")
gmsh.model.addPhysicalGroup(2, [rectangle_tag], -1, "omega")

#..6/8: generate mesh
gmsh.model.mesh.generate(2)

#..7/8: write to file
if (true) gmsh.write("quad_square.msh") end
if (true) gmsh.fltk.run() end

#..8/8: finalize
should_finalize && Gmsh.finalize(); 

### Section 2.4: Inductor Geometry using Open-Cascade 

In [23]:
?gmsh.model.getEntities()

```
gmsh.model.getEntities(dim = -1)
```

Get all the entities in the current model. A model entity is represented by two integers: its dimension (dim == 0, 1, 2 or 3) and its tag (its unique, strictly positive identifier). If `dim` is >= 0, return only the entities of the specified dimension (e.g. points if `dim` == 0). The entities are returned as a vector of (dim, tag) pairs.

Return `dimTags`.

Types:

  * `dimTags`: vector of pairs of integers
  * `dim`: integer


In [46]:
Gmsh.finalize()

In [52]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.setNumber("General.Terminal", 1)
gmsh.option.setNumber("Mesh.MeshSizeMax",3)
gmsh.model.add("inductor")
#defining parameters
dim = 2 
mesh_size = 0.1

#..3/8: generate geometry

#....set length of the air gap 
lg = 5

#....step-1: create and name core domain (no gap in middle leg).. 
out_tag       = gmsh.model.occ.add_rectangle(-27.25, -27.6-lg/2, 0, 54.5, lg+2*27.6)
right_coil    = gmsh.model.occ.add_rectangle(7.422, -20.2-lg/2, 0, 13.178, lg+2*20.2)
left_coil     = gmsh.model.occ.add_rectangle(-20.6, -20.2-lg/2, 0, 13.178, lg+2*20.2)
air_gap_left  = gmsh.model.occ.add_rectangle(-27.25, -lg/2, 0, 6.65, lg)
air_gap_right = gmsh.model.occ.add_rectangle(20.6, -lg/2, 0, 6.65, lg)
core_tag      = gmsh.model.occ.cut([(2, out_tag)],  [(2, right_coil),(2, left_coil),(2,air_gap_left),(2,air_gap_right)])
gmsh.model.setEntityName(dim,core_tag[1][1][2],"core")

#....step-2: create and name right coil domain
right_coil_tag = gmsh.model.occ.add_rectangle(7.422, -20.2-lg/2, 0, 13.178, lg+2*20.2)
gmsh.model.setEntityName(dim,right_coil_tag,"right_coil")

#....step-3: create and name left coil domain
left_coil_tag  = gmsh.model.occ.add_rectangle(-20.6, -20.2-lg/2, 0, 13.178, lg+2*20.2)
gmsh.model.setEntityName(dim,left_coil_tag,"left_coil")

#....step-4: create and name air domain 
gmsh.model.occ.add_rectangle(-50, -50, 0, 100, 100)
gmsh.model.setEntityName(dim,air_tag,"air")

#....step-5: synchronize  model (required here in order for model.getEntities() to work properly) 
gmsh.model.occ.synchronize()

#....step-6: get entities  
entities = gmsh.model.getEntities(2) # see documentation of getEnties 

#....step-7: subtract core from air domain 
for e in entities
    dim  = e[1]
    tag  = e[2]
    name = gmsh.model.getEntityName(dim,tag)

    if ((name=="core")||(name=="right_coil")||(name=="left_coil"))
        copy = gmsh.model.occ.copy(e)
        difference = gmsh.model.occ.cut([(dim, air_tag)],e) 
    end 
    
end

#....step-8: synchronize  model
gmsh.model.occ.synchronize()

#..6/8: generate mesh 
gmsh.model.mesh.generate(2)
gmsh.model.mesh.removeDuplicateNodes()
gmsh.model.mesh.renumberNodes()

#..7/8: write mesh to mesh and visualize the mesh  
#..if true, write mesh to file for further processing 
if (false) gmsh.write("inductor.msh") end 
#..if true, visualize mesh through the GUI 
if (true) gmsh.fltk.run() end 

#..8/8: finalize gmsh 
should_finalize && Gmsh.finalize() 


Info    : Meshing 1D...ence - Adding holes                                                                                                       
Info    : [  0%] Meshing curve 42 (Line)
Info    : [ 10%] Meshing curve 43 (Line)
Info    : [ 10%] Meshing curve 44 (Line)
Info    : [ 10%] Meshing curve 45 (Line)
Info    : [ 10%] Meshing curve 46 (Line)
Info    : [ 20%] Meshing curve 47 (Line)
Info    : [ 20%] Meshing curve 48 (Line)
Info    : [ 20%] Meshing curve 49 (Line)
Info    : [ 20%] Meshing curve 50 (Line)
Info    : [ 30%] Meshing curve 51 (Line)
Info    : [ 30%] Meshing curve 52 (Line)
Info    : [ 30%] Meshing curve 53 (Line)
Info    : [ 30%] Meshing curve 54 (Line)
Info    : [ 30%] Meshing curve 55 (Line)
Info    : [ 40%] Meshing curve 56 (Line)
Info    : [ 40%] Meshing curve 57 (Line)
Info    : [ 40%] Meshing curve 58 (Line)
Info    : [ 40%] Meshing curve 59 (Line)
Info    : [ 50%] Meshing curve 60 (Line)
Info    : [ 50%] Meshing curve 61 (Line)
Info    : [ 50%] Meshing curve 66 

false

## Section 5: File Input and Output 

How does output file format differ for 
1. triangles vs. quadrilaterals;
2. first order vs. second order elements using <i>gmsh.model.mesh.setOrder(2)</i>;
3. how to read mesh file 

## Section 6: Iterative over Mesh Elements 

In this section, we discuss the GMSH functions <i>gmsh.open</i>, <i>gmsh.model.mesh.getElements</i> and <i>gmsh.model.mesh.getNodes</i> required to perform finite element computations on the meshes that were generated. 

The function <i>gmsh.open</i> is used to load the mesh file from file. This allows to retrieve information on the mesh.  

<b>Requires more information on the labeling of the elements on the subdomain. Labels are continguously stored over the boundary and the subdomain. The labeling on the subdomain does therefore not start at one. Requires more information of how each node of an element is labeled. This infortmationm is required to retrieve the coordinates of the nodes. </b>The function <i>gmsh.model.mesh.getElements</i> is used to retrieve elements on (part of) the domain or (part of) the boundary. For each element in the mesh, a list of of nodes is provided. The information on which nodes belong to an element is referred to as the mesh connectivity information (provide a small example and a node-element incidence relation as a graph).  

<b>Requires more information on how the coordinates of the nodes are stored. Single list of all nodes contigencioiusly stored as (x[i],y[i],z[i]) for all nodes.</b> The function <i>gmsh.model.mesh.getNodes</i> is used to retrieve the coordinates of the nodes. The coordinates allow to define the shape functions on the element.  

1. loop over triangles or quadrilaterals in the (part of the) subdomain and boundary (and thus the importance of physival groups for the subdomain and the boundaries). (how to adapt the syntax (dim,tags) to multiple tags?);
2. extract information on the Jacobian of the element (as a 3-by-3 matrix for a linear element), the area (one-half the determinant of the Jacobian) from GMSH using the function <i>mesh.getJacobians</i> as demonstrated in the tutorial [x6.jl](https://gitlab.onelab.info/gmsh/gmsh/blob/gmsh_4_15_0/tutorials/julia/x6.jl). Extract information on the quadrature points using the function <i>gmsh.model.mesh.getIntegrationPoints</i>;

**Exercises**
1. extract the number of nodes, edges and triangles in the mesh;  
2. verify that sum of area of all elements is equal to area of entire domain;
3. assume that each element (triangle) contributes to the global matrix by a 3-by-3 matrix of all ones. Compute the global matrix by a loop over elements. Initialize the matrix as a sparse matrix. Create a function. Verify that the function is type-stable. Scale matrix per element with the area of the element. Implement stiffness matrix and mass matrix;
4. handle boundary conditions in the stiffness matrix;
5. define right-hand side vector and solve the linear system;  

In [61]:
?gmsh.model.mesh.getElements

```
gmsh.model.mesh.getElements(dim = -1, tag = -1)
```

Get the elements classified on the entity of dimension `dim` and tag `tag`. If `tag` < 0, get the elements for all entities of dimension `dim`. If `dim` and `tag` are negative, get all the elements in the mesh. `elementTypes` contains the MSH types of the elements (e.g. `2` for 3-node triangles: see `getElementProperties` to obtain the properties for a given element type). `elementTags` is a vector of the same length as `elementTypes`; each entry is a vector containing the tags (unique, strictly positive identifiers) of the elements of the corresponding type. `nodeTags` is also a vector of the same length as `elementTypes`; each entry is a vector of length equal to the number of elements of the given type times the number N of nodes for this type of element, that contains the node tags of all the elements of the given type, concatenated: [e1n1, e1n2, ..., e1nN, e2n1, ...].

Return `elementTypes`, `elementTags`, `nodeTags`.

Types:

  * `elementTypes`: vector of integers
  * `elementTags`: vector of vectors of sizes
  * `nodeTags`: vector of vectors of sizes
  * `dim`: integer
  * `tag`: integer


In [101]:
#..1/6: Finalize gmsh
should_finalize = Gmsh.initialize()

#..2/6: Read mesh from file
gmsh.open("square.msh")

#..3/6: Get the subface mesh entity 
# Get all the elementary entities in the model, as a vector of (dimension, tag) pairs
# In case of square tutorial, return dim = 2 and tag = 1 
# or entities[1] = (2,1)
entities = gmsh.model.getEntities(2)

#..4/6: Get elements 
elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(2, 1)

#..5/6: Loop over elements - requires more explanation 
tri = 3
for (i,elemtag) in enumerate(elemTags[1])
   offset = tri*(i-1)
   idx = offset+1:offset+tri 
   inode = elemNodeTags[1][idx]
   if(false)
     println("  elemtag = ",elemtag)  
     println("    elemnode1tag = ",inode[1]) 
     println("    elemnode2tag = ",inode[2]) 
     println("    elemnode3tag = ",inode[3])     
    end 
end 

#..6/6: finalize gmsh 
should_finalize && Gmsh.finalize() 

Info    : Reading 'square.msh'...
Info    : 9 entities
Info    : 142 nodes
Info    : 282 elements
Info    : Done reading 'square.msh'


false

In [113]:
# assume a mesh with e.g. 10 nodes ansd thus 30 (x,y,z) tupples
# then the x-coordinate of the inode is stored in entry 3*(inode-1)+1 
# extend above to include the  information on the nodes 
#..1/6: Finalize gmsh
should_finalize = Gmsh.initialize()

#..2/6: Read mesh from file
gmsh.open("square.msh")

#..3/6: Get the subface mesh entity 
# Get all the elementary entities in the model, as a vector of (dimension, tag) pairs
# In case of square tutorial, return dim = 2 and tag = 1 
# or entities[1] = (2,1)
entities = gmsh.model.getEntities(2)

#..4/6: Get elements 
elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(2, 1)

#..5/6: Get nodes  
nodeTags, node_coord, _ = gmsh.model.mesh.getNodes()
xnode_coord = node_coord[1:3:end]; ynode_coord = node_coord[2:3:end]; 

#..6/6: Loop over elements 
tri = 3
for (i,elemtag) in enumerate(elemTags[1])
   offset = tri*(i-1)
   idx = offset+1:offset+tri 
   inode = elemNodeTags[1][idx]
   xnode = node_coord[3*(inode.-1).+1]; ynode = node_coord[3*(inode.-1).+2]
   if(true)
     println("  elemtag = ",elemtag)  
     println("    elemnode1tag = ",inode[1]) 
     println("      (xnode1,ynode1) = ",xnode[1], ynode[1])        
     println("    elemnode2tag = ",inode[2]) 
     println("      (xnode2,ynode2) = ",xnode[2], ynode[2])                
     println("    elemnode3tag = ",inode[3])    
     println("      (xnode3,ynode3) = ",xnode[3], ynode[3])        
    end 
end 

#..7/7: finalize gmsh 
should_finalize && Gmsh.finalize() 

Info    : Reading 'square.msh'...
Info    : 9 entities
Info    : 142 nodes
Info    : 282 elements
Info    : Done reading 'square.msh'
  elemtag = 41
    elemnode1tag = 72
      (xnode1,ynode1) = 0.70445421856617460.4836609274158978
    elemnode2tag = 81
      (xnode2,ynode2) = 0.75353581093975050.398725467089393
    elemnode3tag = 102
      (xnode3,ynode3) = 0.81679561187383240.4899817334731326
  elemtag = 42
    elemnode1tag = 122
      (xnode1,ynode1) = 0.1890575670468490.2940842362157939
    elemnode2tag = 76
      (xnode2,ynode2) = 0.30000000000293470.3071796769753509
    elemnode3tag = 124
      (xnode3,ynode3) = 0.25039523771713310.3884573638917699
  elemtag = 43
    elemnode1tag = 115
      (xnode1,ynode1) = 0.81504074182015210.8183024367054095
    elemnode2tag = 49
      (xnode2,ynode2) = 0.91037038750433790.758672565470732
    elemnode3tag = 120
      (xnode3,ynode3) = 0.91675370642540570.8508718537559957
  elemtag = 44
    elemnode1tag = 106
      (xnode1,ynode1) = 0.18563431

#### Attempt to understand more 
This is parts of the extended tutorial [x1.jl](https://gitlab.onelab.info/gmsh/gmsh/blob/gmsh_4_15_2/tutorials/julia/x1.jl).

In [66]:
#..1/6: Finalize gmsh
should_finalize = Gmsh.initialize()

#..2/6: Read mesh from file
gmsh.open("square.msh")

# Get all the elementary entities in the model, as a vector of (dimension, tag)
# pairs:
entities = gmsh.model.getEntities()

for e in entities
    # Dimension and tag of the entity:
    dim = e[1]
    tag = e[2]

    # Mesh data is made of `elements' (points, lines, triangles, ...), defined
    # by an ordered list of their `nodes'. Elements and nodes are identified by
    # `tags' as well (strictly positive identification numbers), and are stored
    # ("classified") in the model entity they discretize. Tags for elements and
    # nodes are globally unique (and not only per dimension, like entities).

    # A model entity of dimension 0 (a geometrical point) will contain a mesh
    # element of type point, as well as a mesh node. A model curve will contain
    # line elements as well as its interior nodes, while its boundary nodes will
    # be stored in the bounding model points. A model surface will contain
    # triangular and/or quadrangular elements and all the nodes not classified
    # on its boundary or on its embedded entities. A model volume will contain
    # tetrahedra, hexahedra, etc. and all the nodes not classified on its
    # boundary or on its embedded entities.

    # Get the mesh nodes for the entity (dim, tag):
    nodeTags, nodeCoords, nodeParams = gmsh.model.mesh.getNodes(dim, tag)

    # Get the mesh elements for the entity (dim, tag):
    elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(dim, tag)

    # Elements can also be obtained by type, by using `getElementTypes()'
    # followed by `getElementsByType()'.

    # Let's print a summary of the information available on the entity and its
    # mesh.

    # * Type and name of the entity:
    type = gmsh.model.getType(dim, tag)
    name = gmsh.model.getEntityName(dim, tag)
    if length(name) > 0
        name *= " "
    end
    println("Entity ", name, e, " of type ", type)

    # * Number of mesh nodes and elements:
    numElem = sum(length(i) for i in elemTags; init=0)
    println(" - Mesh has ", length(nodeTags), " nodes and ", numElem,
            " elements")

    # * Upward and downward adjacencies:
    up, down = gmsh.model.getAdjacencies(dim, tag)
    if length(up) > 0
        println(" - Upward adjacencies: ", up)
    end
    if length(down) > 0
        println(" - Downward adjacencies: ", down)
    end

    # * Does the entity belong to physical groups?
    physicalTags = gmsh.model.getPhysicalGroupsForEntity(dim, tag)
    if length(physicalTags) > 0
        s = ""
        for p in physicalTags
            n = gmsh.model.getPhysicalName(dim, p)
            if n != ""
                n *= " "
            end
            s *= n * '(' * string(dim) * ", " * string(p) * ") "
        end
        println(" - Physical groups: " * s)
    end
 
    # * Is the entity a partition entity? If so, what is its parent entity?
    partitions = gmsh.model.getPartitions(dim, tag)
    if length(partitions) > 0
        println(" - Partition tags: ", partitions, " - parent entity ",
                gmsh.model.getParent(dim, tag))
    end

    # * List all types of elements making up the mesh of the entity:
    for t in elemTypes
        name, dim, order, numv, parv, _ = gmsh.model.mesh.getElementProperties(
            t)
        println(" - Element type: ", name, ", order ", order, " (",
                numv, " nodes in param coord: ", parv, ")")
    end
end

#..6/6: Finalize gmsh
should_finalize && Gmsh.finalize()

Info    : Reading 'square.msh'...
Info    : 9 entities
Info    : 142 nodes
Info    : 282 elements
Info    : Done reading 'square.msh'
Entity (0, 1) of type Point
 - Mesh has 1 nodes and 0 elements
 - Upward adjacencies: Int32[1, 4]
Entity (0, 2) of type Point
 - Mesh has 1 nodes and 0 elements
 - Upward adjacencies: Int32[1, 2]
Entity (0, 3) of type Point
 - Mesh has 1 nodes and 0 elements
 - Upward adjacencies: Int32[2, 3]
Entity (0, 4) of type Point
 - Mesh has 1 nodes and 0 elements
 - Upward adjacencies: Int32[3, 4]
Entity (1, 1) of type Discrete curve
 - Mesh has 9 nodes and 10 elements
 - Upward adjacencies: Int32[1]
 - Downward adjacencies: Int32[1, 2]
 - Physical groups: bottom (1, 1) 
 - Element type: Line 2, order 1 (2 nodes in param coord: [-1.0, 1.0])
Entity (1, 2) of type Discrete curve
 - Mesh has 9 nodes and 10 elements
 - Upward adjacencies: Int32[1]
 - Downward adjacencies: Int32[2, 3]
 - Physical groups: right (1, 2) 
 - Element type: Line 2, order 1 (2 nodes in param

false

Loop over elements by loop over elemTags? 

In [68]:
#..1/6: Finalize gmsh
should_finalize = Gmsh.initialize()

#..2/6: Read mesh from file
gmsh.open("square.msh")

# Get all the elementary entities in the model, as a vector of (dimension, tag)
# pairs:
entities = gmsh.model.getEntities(2)

for e in entities
    # Dimension and tag of the entity:
    dim = e[1]
    tag = e[2]

    # Get the mesh nodes for the entity (dim, tag):
    nodeTags, nodeCoords, nodeParams = gmsh.model.mesh.getNodes(dim, tag)

    # Get the mesh elements for the entity (dim, tag):
    elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(dim, tag)

    # * Type and name of the entity:
    type = gmsh.model.getType(dim, tag)
    name = gmsh.model.getEntityName(dim, tag)
    if length(name) > 0
        name *= " "
    end
    println("Entity ", name, e, " of type ", type)

    # * Number of mesh nodes and elements:
    numElem = sum(length(i) for i in elemTags; init=0)
    println(" - Mesh has ", length(nodeTags), " nodes and ", numElem,
            " elements")

    # * Does the entity belong to physical groups?
    physicalTags = gmsh.model.getPhysicalGroupsForEntity(dim, tag)
    if length(physicalTags) > 0
        s = ""
        for p in physicalTags
            n = gmsh.model.getPhysicalName(dim, p)
            if n != ""
                n *= " "
            end
            s *= n * '(' * string(dim) * ", " * string(p) * ") "
        end
        println(" - Physical groups: " * s)
    end
 
    # * List all types of elements making up the mesh of the entity:
    for t in elemTypes
        name, dim, order, numv, parv, _ = gmsh.model.mesh.getElementProperties(
            t)
        println(" - Element type: ", name, ", order ", order, " (",
                numv, " nodes in param coord: ", parv, ")")
    end
end

#..6/6: Finalize gmsh
should_finalize && Gmsh.finalize()

Info    : Reading 'square.msh'...
Info    : 9 entities
Info    : 142 nodes
Info    : 282 elements
Info    : Done reading 'square.msh'
Entity (2, 1) of type Discrete surface
 - Mesh has 102 nodes and 242 elements
 - Physical groups: omega (2, 5) 
 - Element type: Triangle 3, order 1 (3 nodes in param coord: [0.0, 0.0, 1.0, 0.0, 0.0, 1.0])


false

elemTags: a Vector of Vector of UInt64: elemTaga[1] holds the tags of elements in first entity

how to use elemNodeTags? 

Info    : Reading 'square.msh'...
Info    : 9 entities
Info    : 142 nodes
Info    : 282 elements
Info    : Done reading 'square.msh'
  elemtag = 41
    elemnode1tag = 72
    elemnode2tag = 81
    elemnode3tag = 102
  elemtag = 42
    elemnode1tag = 122
    elemnode2tag = 76
    elemnode3tag = 124
  elemtag = 43
    elemnode1tag = 115
    elemnode2tag = 49
    elemnode3tag = 120
  elemtag = 44
    elemnode1tag = 106
    elemnode2tag = 52
    elemnode3tag = 121
  elemtag = 45
    elemnode1tag = 52
    elemnode2tag = 106
    elemnode3tag = 122
  elemtag = 46
    elemnode1tag = 49
    elemnode2tag = 115
    elemnode3tag = 134
  elemtag = 47
    elemnode1tag = 96
    elemnode2tag = 122
    elemnode3tag = 124
  elemtag = 48
    elemnode1tag = 47
    elemnode2tag = 86
    elemnode3tag = 107
  elemtag = 49
    elemnode1tag = 52
    elemnode2tag = 122
    elemnode3tag = 125
  elemtag = 50
    elemnode1tag = 93
    elemnode2tag = 42
    elemnode3tag = 95
  elemtag = 51
    elemnode1tag = 89
   

726-element Vector{Int64}:
  72
  81
 102
 122
  76
 124
 115
  49
 120
 106
  52
 121
  52
   ⋮
 106
 128
 137
 126
  87
 142
  87
 130
 142
 130
  51
 142

In [ ]:
#..1/6: Finalize gmsh
should_finalize = Gmsh.initialize()

#..2/6: Read mesh from file
gmsh.open("square.msh")

# 2. Haal ALLE knopen (nodes) op van het oppervlak (dim=2, tag=1)
# Note: Julia gebruikt 1-based indexing, maar Gmsh tags blijven UInt64 integers
nodeTags, coord, _ = gmsh.model.mesh.getNodes(2, 1)

# 3. Maak een Dictionary: Dict(nodeTag => [x, y, z])
node_coords = Dict{UInt64, Vector{Float64}}()
for i in 1:length(nodeTags)
    idx = (i - 1) * 3 + 1
    node_coords[nodeTags[i]] = [coord[idx], coord[idx+1], coord[idx+2]]
end

# 4. Haal de elementen op van het oppervlak
elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(2, 1)

# 5. Koppel de coördinaten aan de elementen
for i in 1:length(elemTypes)
    elem_type = elemTypes[i]
    
    # Vraag eigenschappen op, zoals het aantal knopen per element (numNodes)
    _, _, _, numNodes, _, _ = gmsh.model.mesh.getElementProperties(elem_type)
    
    tags_van_type = elemTags[i]
    knopen_van_type = elemNodeTags[i]
    
    println("\n--- Element Type ID: $elem_type ($numNodes knopen per element) ---")
    
    # Loop door elk individueel element van dit type
    for j in 1:length(tags_van_type)
        elem_tag = tags_van_type[j]
        
        # Bereken de indices voor de knopen van dit specifieke element
        start_idx = (j - 1) * numNodes + 1
        end_idx = start_idx + numNodes - 1
        sub_node_tags = knopen_van_type[start_idx:end_idx]
        
        println("Element ID $elem_tag bestaat uit knopen: $sub_node_tags")
        
        # Haal de coördinaten op via de Dict
        for node_tag in sub_node_tags
            xyz = node_coords[node_tag]
            @printf("  -> Knoop %d: X=%.2f, Y=%.2f, Z=%.2f\n", node_tag, xyz[1], xyz[2], xyz[3])
        end
    end
end

## Section 7: Concluding Remarks 

1. combining triangular and quadrilateral elements in the mesh; 
3. transfinite meshing allowing refinewment along edges; 
4. 3D mesh generating, extrude, revolve; 

In [88]:
726 / 3

242.0